In [3]:
import torch 
import torch.nn as nn

class CausualAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, bias=False):
        super().__init__()

        self.wq = nn.Linear(d_in, d_out, bias)
        self.wk = nn.Linear(d_in, d_out, bias)
        self.wv = nn.Linear(d_in, d_out, bias)

        self.drop_out = nn.Dropout(dropout)

        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    def forward(self, x):
        batch_size, num_tokens, d_in=x.shape
        # Tensor의 크기를 본다, batch 사이크 token의 개수, token의 Dimension

        keys = self.wk(x)
        quries = self.wq(x)
        values = self.wv(x)

        #Query와 Key를 내적해서 토큰 간의 관련성을 구한다.
        atten_scores = queries @ keys.transpose(1,2)

        #Mask가 1인 위티를 -무한대로 채운다.
        atten_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            -torch.inf
        )
